# Exportação da camada Gold para CSV

Este notebook exporta as sete tabelas Gold do catálogo para CSV, com cabeçalho e separador ponto e vírgula.

Os arquivos são gravados no Unity Catalog Volume em uma pasta por tabela.
Uma execução nova sobrescreve a exportação anterior de cada tabela.

O Volume de destino deve ter acesso restrito às pessoas e serviços autorizados.

In [0]:

# Configuração do Ambiente e Volume

CATALOG = "workspace"
SCHEMA_GOLD = "gold"

# Cria o Volume no Databricks via Spark SQL usando as variáveis
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}.gold_export_volume")

# Caminho raiz para a exportação dos CSVs e do Manifesto
PATH_EXPORT = f"/Volumes/{CATALOG}/{SCHEMA_GOLD}/gold_export_volume/export_csv"

In [0]:


TABELAS_GOLD = [
    "diagnostico_receita",
    "indicadores_dec_fec",
    "fila_inspecao",
    "visao_360_uc",
    "consumo_mensal_uc",
    "consumo_receita_bairro",
    "variacao_consumo_uc"
]

print(f"Iniciando exportação de {len(TABELAS_GOLD)} tabelas Gold.")
print(f"Destino: {PATH_EXPORT}")

# Verifica todas as tabelas antes de começar, para não terminar
# com uma exportação parcial por nome incorreto ou tabela ausente.
tabelas_ausentes = [
    nome
    for nome in TABELAS_GOLD
    if not spark.catalog.tableExists(
        f"{CATALOG}.{SCHEMA_GOLD}.{nome}"
    )
]

if tabelas_ausentes:
    raise ValueError(
        f"Tabelas Gold não encontradas: {tabelas_ausentes}"
    )

for nome_tabela in TABELAS_GOLD:
    tabela_completa = f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}"
    caminho_saida = f"{PATH_EXPORT}/{nome_tabela}"

    df = spark.table(tabela_completa)

    (
        df.write
        .format("csv")
        .mode("overwrite")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "UTF-8")
        .option("quoteAll", "true")
        .save(caminho_saida)
    )

    print(f"[OK] {tabela_completa} -> {caminho_saida}")

print("Exportação das sete tabelas Gold concluída.")

testo manifesto 


###  Geração Automática do Manifesto de Exportação (`manifest.json`)

Esta etapa é responsável por registrar os metadados de rastreabilidade do lote de dados exportado da camada **Gold**, criando o arquivo `manifest.json` no Volume do Databricks. 

O script identifica dinamicamente o tipo de execução para garantir governança e auditoria tanto em ambiente de testes manuais quanto em produção automatizada:

---

####  Funcionamento do Código
1. **Captura do Run ID de Job (Produção):**
   * O código consulta as *tags* de contexto do Databricks em busca de um `multitaskParentRunId` ou `runId` gerado por uma execução oficial via **Workflows / Jobs**.
2. **Tratamento de Execução Manual (Dev / Testes):**
   * Caso o notebook seja executado manualmente (botão *Run*), o script identifica o usuário logado e gera uma chave de lote dinâmica baseada em timestamp (`batch_<usuario>_<YYYYMMDDTHHMMSS>`), evitando IDs fixos ou genéricos.
3. **Mapeamento de Metadados:**
   * Estrutura o JSON contendo o ambiente (`PROD`), a lista das 7 tabelas exportadas, a data/hora exata do processamento e o identificador do lote (`batch_ref`).
4. **Persistência no Volume:**
   * Grava o arquivo no diretório configurado em `PATH_EXPORT` (`/Volumes/workspace/gold/gold_export_volume/export_csv/manifest.json`).

---

####  Saídas Esperadas
* **Via Job:** 
  ```json
  "lote": "10849202394",
  "run_id": "10849202394"
  ```
* **Via Execução Manual no Notebook:** 
  ```json
  "lote": "batch_beatriz_20261027T185800",
  "run_id": "batch_beatriz_20261027T185800"
  ```





In [0]:
import json
import os
import zoneinfo
from datetime import datetime

# 1. Configura o fuso horário de Brasília/Recife (UTC-3)
fuso_br = zoneinfo.ZoneInfo("America/Recife")
data_formatada = datetime.now(fuso_br).strftime("%Y-%m-%d %H:%M:%S")

# 2. Tenta capturar o Run ID do Job ou define o lote para execução manual
run_id_real = None

try:
    # Captura o contexto serializado em JSON (compatível com Spark Connect)
    context_str = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
    context = json.loads(context_str)
    
    # 1ª Tentativa: Se estiver rodando via Job / Workflow
    tags = context.get("tags", {})
    run_id_real = tags.get("multitaskParentRunId") or tags.get("runId") or tags.get("currentRunId")
    
    # 2ª Tentativa: ID da raiz do comando se disponível
    if not run_id_real:
        run_id_real = context.get("currentRunId", {}).get("id")
except Exception:
    pass

# 3ª Tentativa (Garantia para Execução Manual no Spark Connect)
if not run_id_real or run_id_real in ["run_manual_local", "execucao_manual"]:
    try:
        user_email = spark.sql("SELECT current_user()").collect()[0][0]
        user_prefix = user_email.split("@")[0]
    except Exception:
        user_prefix = "beatriz"
    
    # Gera a chave do lote com a data e hora no fuso do Brasil
    timestamp_atual = datetime.now(fuso_br).strftime("%Y%m%dT%H%M%S")
    run_id_real = f"batch_{user_prefix}_{timestamp_atual}"

# 3. Monta a estrutura do manifesto usando a 'data_formatada'
manifesto = {
    "ambiente": "PROD",
    "lote": run_id_real,
    "run_id": run_id_real,
    "data_processamento": data_formatada,
    "data_exportacao_csv": data_formatada,
    "tabelas_exportadas": [
        "diagnostico_receita",
        "indicadores_dec_fec",
        "fila_inspecao",
        "visao_360_uc",
        "consumo_mensal_uc",
        "consumo_receita_bairro",
        "variacao_consumo_uc"
    ],
    "responsavel": "Equipe de Engenharia de Dados (Gold)"
}

# 4. Salva o arquivo manifest.json no Volume
os.makedirs(PATH_EXPORT, exist_ok=True)
caminho_manifesto = os.path.join(PATH_EXPORT, "manifest.json")

with open(caminho_manifesto, "w", encoding="utf-8") as f:
    json.dump(manifesto, f, indent=4, ensure_ascii=False)

print(f"[OK] Manifesto gerado com sucesso!")
print(f" Lote / Run ID gravado: {run_id_real}")
print(f" Data/Hora gravada: {data_formatada}")
print(f" Salvo em: {caminho_manifesto}")